# 🧠 Brain Tumor MRI Segmentation using U-Net + GLCM Texture Features

End-to-end pipeline for segmenting brain tumors from MRI scans, combining a lightweight
**U-Net** architecture with **GLCM (Gray-Level Co-occurrence Matrix)** texture features
and a simulated GAN-style augmentation channel, followed by morphological post-processing
and clinical-style reporting.

See `README.md` for full project details, methodology, and results.

## 1. Setup & Dependencies

In [ ]:
!pip install -q opencv-python scikit-image tensorflow matplotlib scikit-learn kaggle


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from tensorflow.keras.models import Model


## 2. Dataset

This project uses the **LGG MRI Segmentation** dataset (Kaggle: `mateuszbuda/lgg-mri-segmentation`), containing brain MRI slices paired with
manually annotated tumor masks.

### Option A — Kaggle API (recommended for Colab)
```bash
mkdir -p ~/.kaggle
# upload your kaggle.json first, then:
mv kaggle.json ~/.kaggle/
chmod 600 ~/.kaggle/kaggle.json
kaggle datasets download -d mateuszbuda/lgg-mri-segmentation
unzip -q lgg-mri-segmentation.zip
```

### Option B — Local / manual download
Download the dataset manually from Kaggle and place it under `data/kaggle_3m/`
(one folder per patient, each containing `*.tif` images and matching `*_mask.tif` masks).

In [ ]:
DATASET_PATH = "data/kaggle_3m"   # update this path if your data lives elsewhere
IMG_SIZE = 128


## 3. Data Loading

In [ ]:
def load_dataset(path, img_size=128, max_samples=None, tumor_only=False):
    """Load MRI slices and their tumor masks from the LGG dataset folder structure.

    Args:
        path: root dataset folder (one sub-folder per patient).
        img_size: images/masks are resized to (img_size, img_size).
        max_samples: optionally cap the number of loaded samples.
        tumor_only: if True, skip slices that contain no tumor pixels.
    """
    images, masks = [], []

    for patient in os.listdir(path):
        patient_path = os.path.join(path, patient)
        if not os.path.isdir(patient_path):
            continue

        for file in os.listdir(patient_path):
            if file.endswith(".tif") and "_mask" not in file:
                img_path = os.path.join(patient_path, file)
                mask_path = img_path.replace(".tif", "_mask.tif")

                if not os.path.exists(mask_path):
                    continue

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

                if tumor_only and np.sum(mask) == 0:
                    continue

                img = cv2.resize(img, (img_size, img_size)) / 255.0
                mask = cv2.resize(mask, (img_size, img_size))
                mask = (mask > 0).astype(np.float32)

                images.append(img)
                masks.append(mask)

                if max_samples and len(images) >= max_samples:
                    return np.array(images), np.array(masks)

    return np.array(images), np.array(masks)


In [ ]:
images, masks = load_dataset(DATASET_PATH, IMG_SIZE, max_samples=200, tumor_only=True)

print("Images:", images.shape)
print("Masks :", masks.shape)


### Quick sanity check

Visualize one MRI slice next to its ground-truth tumor mask.

![Sample MRI and mask](../images/01_sample_mri_and_mask.png)

In [ ]:
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(images[0], cmap="gray")
plt.title("MRI Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(masks[0], cmap="gray")
plt.title("Tumor Mask")
plt.axis("off")

plt.tight_layout()
plt.show()


## 4. Feature Engineering

Each MRI slice is turned into a **3-channel input** for the network:

1. **Original grayscale MRI** — the raw intensity signal.
2. **Simulated GAN channel** — a lightly noise-augmented version of the image, standing in
   for a GAN-based augmentation/denoising branch.
3. **GLCM texture channel** — a Gray-Level Co-occurrence Matrix *contrast* feature map,
   which captures local texture information useful for distinguishing tumor tissue.


In [ ]:
def glcm_feature_map(image):
    """Compute a GLCM contrast texture map for a single grayscale image."""
    img_uint8 = (image * 255).astype(np.uint8)

    glcm = graycomatrix(
        img_uint8,
        distances=[1],
        angles=[0],
        levels=256,
        symmetric=True,
        normed=True,
    )

    contrast = graycoprops(glcm, "contrast")[0, 0]
    return np.full(image.shape, contrast)


def simulated_gan(image):
    """Lightweight stand-in for a GAN-based augmentation branch (Gaussian noise)."""
    fake = image + np.random.normal(0, 0.05, image.shape)
    return np.clip(fake, 0, 1)


In [ ]:
X, Y = [], []

for img, mask in zip(images, masks):
    gan_img = simulated_gan(img)
    glcm_map = glcm_feature_map(img)

    stacked_input = np.stack([img, gan_img, glcm_map], axis=-1)
    X.append(stacked_input)
    Y.append(mask[..., np.newaxis])

X = np.array(X)
Y = np.array(Y)

print("Final X shape:", X.shape)
print("Final Y shape:", Y.shape)


## 5. Train / Validation Split

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, Y_train.shape)
print("Val  :", X_val.shape, Y_val.shape)


## 6. Model — Lightweight U-Net

A compact encoder-decoder (U-Net style) with skip connections between the encoder and
decoder paths, sized for fast experimentation on 128×128 inputs.

In [ ]:
def build_unet(input_shape=(128, 128, 3)):
    inputs = Input(input_shape)

    c1 = Conv2D(16, 3, activation="relu", padding="same")(inputs)
    p1 = MaxPooling2D()(c1)

    c2 = Conv2D(32, 3, activation="relu", padding="same")(p1)
    p2 = MaxPooling2D()(c2)

    c3 = Conv2D(64, 3, activation="relu", padding="same")(p2)

    u1 = UpSampling2D()(c3)
    m1 = concatenate([u1, c2])
    c4 = Conv2D(32, 3, activation="relu", padding="same")(m1)

    u2 = UpSampling2D()(c4)
    m2 = concatenate([u2, c1])
    c5 = Conv2D(16, 3, activation="relu", padding="same")(m2)

    outputs = Conv2D(1, 1, activation="sigmoid")(c5)

    return Model(inputs, outputs)


tf.keras.backend.clear_session()
model = build_unet((IMG_SIZE, IMG_SIZE, 3))

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 7. Training

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
)

history = model.fit(
    X_train,
    Y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.1,
    callbacks=[early_stop],
)


In [ ]:
os.makedirs("../models", exist_ok=True)
model.save("../models/brain_tumor_unet.keras")


## 8. Inference on the Validation Set

In [ ]:
pred = model.predict(X_val)
pred_mask = (pred > 0.15).astype(np.uint8)   # threshold tuned empirically on this dataset

print("Min prediction:", np.min(pred))
print("Max prediction:", np.max(pred))


### Visual check — MRI, ground truth, and raw prediction

![Prediction comparison](../images/02_prediction_comparison.png)

In [ ]:
index = 0

plt.figure(figsize=(15, 5))

plt.subplot(1, 4, 1)
plt.imshow(X_val[index][:, :, 0], cmap="gray")
plt.title("Input MRI")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(Y_val[index].squeeze(), cmap="gray")
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(pred_mask[index].squeeze(), cmap="gray")
plt.title("Predicted Mask")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(X_val[index][:, :, 0], cmap="gray")
plt.imshow(pred_mask[index].squeeze(), cmap="jet", alpha=0.4)
plt.title("Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()


## 9. Evaluation Metrics

Standard segmentation metrics computed pixel-wise across the validation set:
**Dice coefficient, IoU (Jaccard index), pixel accuracy, precision, recall, and F1-score.**

In [ ]:
def dice_coefficient(y_true, y_pred):
    smooth = 1e-6
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred) + smooth)


def iou_score(y_true, y_pred):
    smooth = 1e-6
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)


def pixel_accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / y_true.size


def precision_score(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    return TP / (TP + FP + 1e-6)


def recall_score(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP / (TP + FN + 1e-6)


def f1_score(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return 2 * (precision * recall) / (precision + recall + 1e-6)


In [ ]:
dice = dice_coefficient(Y_val, pred_mask)
iou = iou_score(Y_val, pred_mask)
accuracy = pixel_accuracy(Y_val, pred_mask)
precision = precision_score(Y_val, pred_mask)
recall = recall_score(Y_val, pred_mask)
f1 = f1_score(Y_val, pred_mask)

print("Dice Score     :", round(dice, 4))
print("IoU Score      :", round(iou, 4))
print("Pixel Accuracy :", round(accuracy, 4))
print("Precision      :", round(precision, 4))
print("Recall         :", round(recall, 4))
print("F1 Score       :", round(f1, 4))


## 10. Post-Processing

Raw predictions can contain small spurious "false-positive" blobs — very common in
medical image segmentation. We clean these up in two steps:

1. **Morphological opening** to remove small noisy regions.
2. **Keep-largest-connected-component** to retain only the primary tumor region.

In [ ]:
def keep_largest_region(mask):
    """Keep only the largest connected component in a binary mask."""
    mask = mask.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    if num_labels <= 1:
        return mask

    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    cleaned_mask = np.zeros_like(mask)
    cleaned_mask[labels == largest_label] = 1
    return cleaned_mask


def process_batch(pred_mask):
    processed = []
    for i in range(pred_mask.shape[0]):
        mask_2d = pred_mask[i].squeeze()
        cleaned = keep_largest_region(mask_2d)
        processed.append(cleaned)
    return np.array(processed)


pred_mask_final = process_batch(pred_mask)


In [ ]:
dice = dice_coefficient(Y_val.squeeze(), pred_mask_final)
iou = iou_score(Y_val.squeeze(), pred_mask_final)

print("Dice After Post-Processing:", round(dice, 4))
print("IoU After Post-Processing :", round(iou, 4))


### Tumor overlay on the original MRI

![Tumor overlay](../images/03_tumor_overlay.png)

In [ ]:
index = 0

plt.figure(figsize=(6, 6))
plt.imshow(X_val[index].squeeze()[..., 0] if X_val[index].ndim == 3 else X_val[index].squeeze(), cmap="gray")
plt.imshow(pred_mask_final[index], cmap="jet", alpha=0.4)
plt.title("Tumor Overlay on MRI (Post-Processed)")
plt.axis("off")
plt.show()


## 11. Tumor Area & Clinical Summary

Tumor area gives an approximate sense of **tumor burden** — how much of the imaged brain
tissue is occupied by tumor in a given slice, which can support:

- **Disease severity estimation** (small / moderate / large lesion)
- **Treatment monitoring** across scans taken over time

> ⚠️ This is an AI-generated estimate for demonstration purposes only — not a substitute
> for clinical diagnosis.

In [ ]:
pixel_spacing = 0.5  # mm per pixel (dataset-dependent — adjust to your scanner metadata)

tumor_pixels = np.sum(pred_mask_final[index] == 1)
total_pixels = pred_mask_final[index].size
tumor_area_mm2 = tumor_pixels * (pixel_spacing ** 2)
tumor_percent = (tumor_pixels / total_pixels) * 100

print("Tumor Area      :", round(tumor_area_mm2, 2), "mm^2")
print("Tumor Percentage: {:.2f}%".format(tumor_percent))


In [ ]:
def clinical_report():
    print("\n" + "=" * 60)
    print("        AI-ASSISTED BRAIN TUMOR SEGMENTATION REPORT")
    print("=" * 60)

    print("\nTumor Burden Analysis:")
    print("   Estimated Tumor Percentage : {:.2f}%".format(tumor_percent))

    print("\nQuantitative Performance Metrics:")
    print("   Dice Coefficient      : {:.4f}".format(dice))
    print("   IoU (Jaccard Index)   : {:.4f}".format(iou))
    print("   Pixel Accuracy        : {:.4f}".format(accuracy))
    print("   Precision             : {:.4f}".format(precision))
    print("   Recall (Sensitivity)  : {:.4f}".format(recall))
    print("   F1 Score              : {:.4f}".format(f1))

    print("\nClinical Interpretation:")
    if dice > 0.85:
        print("   - Excellent tumor boundary delineation.")
    elif dice > 0.75:
        print("   - Good segmentation performance.")
    else:
        print("   - Moderate segmentation performance.")

    if tumor_percent > 10:
        print("   - Significant tumor presence detected.")
    else:
        print("   - Tumor burden appears limited.")

    print("\nNote: AI-generated estimation. Clinical validation required.")
    print("=" * 60)


clinical_report()


## 12. Results Visualization

#### Metrics overview
![Metrics bar chart](../images/04_metrics_bar_chart.png)
![Metrics line chart](../images/05_metrics_line_chart.png)

#### Effect of post-processing
![Before vs after post-processing](../images/06_before_after_postprocessing.png)

In [ ]:
metric_names = ["Dice", "IoU", "Accuracy", "Precision", "Recall", "F1-Score"]
metric_values = [dice, iou, accuracy, precision, recall, f1]

plt.figure(dpi=150)
bars = plt.bar(metric_names, metric_values)
plt.ylim(0, 1)
plt.xlabel("Evaluation Metrics")
plt.ylabel("Score")
plt.title("Performance Evaluation of the Segmentation Model")

for i, v in enumerate(metric_values):
    plt.text(i, v + 0.02, round(v, 3), ha="center")

plt.tight_layout()
plt.savefig("../images/evaluation_metrics.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
dice_before = dice_coefficient(Y_val.squeeze(), pred_mask)
iou_before = iou_score(Y_val.squeeze(), pred_mask)

dice_after = dice_coefficient(Y_val.squeeze(), pred_mask_final)
iou_after = iou_score(Y_val.squeeze(), pred_mask_final)

before = [dice_before, iou_before]
after = [dice_after, iou_after]

labels = ["Dice", "IoU"]
x = np.arange(len(labels))
width = 0.35

plt.figure(dpi=150)
plt.bar(x - width / 2, before, width, label="Before Post-Processing")
plt.bar(x + width / 2, after, width, label="After Post-Processing")
plt.ylim(0, 1)
plt.xticks(x, labels)
plt.ylabel("Score")
plt.title("Performance Improvement After Post-Processing")
plt.legend()
plt.tight_layout()
plt.savefig("../images/comparison_graph.png", dpi=150, bbox_inches="tight")
plt.show()


## Summary

This notebook demonstrates a full, reproducible pipeline for AI-assisted brain tumor
segmentation: data loading → texture/GAN-style feature engineering → U-Net training →
morphological post-processing → quantitative evaluation → clinical-style reporting.

See the project `README.md` for a discussion of results, limitations, and possible
extensions (e.g. a real GAN augmentation branch, deeper U-Net/attention variants,
3D volumetric segmentation, or multi-class tumor sub-region labeling).